# Kiểm tra toàn bộ lưới GridSearchCV

Phụ lục độc lập của Notebook 02. Đọc `data/processed/v7_preprocessed.csv`,
chia cùng train/test và năm fold, rồi **chạy lại tất cả cấu hình BoW/TF-IDF ×
Naive Bayes/SVM × hai nhánh từ dừng × n-gram × alpha/C**. Điểm ở đây chỉ là
**validation trong train**; test độc lập không dùng để lựa chọn.

Notebook này không sửa Notebook 02, mã nguồn, CSV đầu vào hay mô hình đã lưu.
Hai CSV kiểm toán được lưu riêng ở `artifacts/grid_audit/`.

## 1. Chuẩn bị

Chọn interpreter `.venv/Scripts/python.exe` làm kernel. Mở từ thư mục project
và chạy từ trên xuống. Các lớp/hàm được import từ mã hiện tại của nhóm.

In [1]:
import os, sys, tempfile
from pathlib import Path
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from joblib import Memory
from sklearn.model_selection import GridSearchCV, ParameterGrid

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "pipeline/__init__.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Mở notebook từ trong thư mục project.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from pipeline.modeling.text_features import FEATURES, NGRAMS
from pipeline.modeling.classifiers import MODELS, NB_ALPHAS, SVM_CS
from pipeline.modeling.experiments import make_pipe, CV_SCORING
from pipeline.modeling.training_utils import split_train_test, grouped_folds

SEED = 42
np.random.seed(SEED)
N_JOBS = int(os.environ.get("SPAMHAM_N_JOBS", "2"))
DATA_PATH = ROOT / "data/processed/v7_preprocessed.csv"
OUT = ROOT / "artifacts/grid_audit"
OUT.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", 30)
print("Python", sys.version.split()[0], "| sklearn", sklearn.__version__)
print("Dữ liệu:", DATA_PATH, "| jobs:", N_JOBS)

Python 3.12.14 | sklearn 1.6.1
Dữ liệu: C:\KH086\PT&KP DLVB\PROJECT\ptit-facebook-spam-ham\data\processed\v7_preprocessed.csv | jobs: 2


## 2. Dữ liệu và phép chia

CSV này do Notebook 02 tạo; ID, nhãn và `cleaned_text` giữ cùng thứ tự
với V7 gốc. `split_train_test` giữ nội dung trùng cùng phía; `grouped_folds`
chia năm lượt **chỉ trong train**. Validation không được fit trong lượt tương ứng.

In [2]:
data = pd.read_csv(DATA_PATH, keep_default_na=False, dtype={"id": str})
assert data.id.is_unique and set(data.label.str.strip().str.lower()) == {"ham", "spam"}
y_all = data.label.str.strip().str.lower().eq("spam").astype(int).to_numpy()
TEXT = {"giữ stop words": data.tokens_keep_stopwords,
        "có loại stop words": data.tokens_remove_stopwords}
keys, train_idx, test_idx = split_train_test(data.cleaned_text, y_all)
y_train = y_all[train_idx]
FOLDS = grouped_folds(keys[train_idx], y_train)
assert not set(keys[train_idx]) & set(keys[test_idx])
display(pd.DataFrame({"Tập": ["Train", "Test"],
                      "Số mẫu": [len(train_idx), len(test_idx)],
                      "Spam": [int(y_train.sum()), int(y_all[test_idx].sum())]}))
display(pd.DataFrame([{"Fold": i + 1, "Train": len(tr), "Validation": len(va),
                       "Spam validation": int(y_train[va].sum())}
                      for i, (tr, va) in enumerate(FOLDS)]))

,Tập,Số mẫu,Spam
0,Train,30759,4375
1,Test,7474,951


,Fold,Train,Validation,Spam validation
0,1,24815,5944,764
1,2,24327,6432,1187
2,3,24554,6205,856
3,4,24529,6230,789
4,5,24811,5948,779


## 3. Grid là danh sách giá trị cần thử

`(1,1)` = chỉ từ đơn; `(1,2)` = từ đơn **và** cặp từ liền nhau.
`make_pipe` trả Pipeline chưa fit, `grid` chứa các danh sách giá trị.
`ParameterGrid` liệt kê tích Descartes của chúng; nó chưa huấn luyện.

Ba danh sách dưới đây là **chỗ sửa khi muốn thử thêm cấu hình**. Ví dụ thêm
`0.5` vào `AUDIT_CS`, rồi chạy lại từ cell này trở xuống. Giữ nguyên mặc định
để khớp Notebook 02. SVM cơ sở dùng `class_weight=None`; S1–S3 là bước sau.

In [3]:
AUDIT_NGRAMS = list(NGRAMS)       # [(1,1), (1,2)]
AUDIT_ALPHAS = list(NB_ALPHAS)  # [0.1, 0.3, 1.0]
AUDIT_CS = list(SVM_CS)         # [0.1, 0.3, 1.0, 3.0]

def audit_pipe(feature, algorithm, cache=None):
    pipe, grid, _ = make_pipe(feature, algorithm, seed=SEED,
                              cache=cache, n_jobs=N_JOBS, class_weight=None)
    grid["vec__ngram_range"] = AUDIT_NGRAMS
    if algorithm == "Naive Bayes":
        grid["clf__alpha"] = AUDIT_ALPHAS
    else:
        grid["clf__C"] = AUDIT_CS
    return pipe, grid

plan = []
for pre in TEXT:
    for feature in FEATURES:
        for algorithm in MODELS:
            _, grid = audit_pipe(feature, algorithm)
            plan.append({"Tiền xử lý": pre, "Đặc trưng": feature,
                         "Thuật toán": algorithm,
                         "Cấu hình": len(ParameterGrid(grid)),
                         "Lượt fit CV": len(ParameterGrid(grid)) * len(FOLDS)})
plan = pd.DataFrame(plan)
display(plan)
print("Tổng", int(plan["Cấu hình"].sum()), "cấu hình,",
      int(plan["Lượt fit CV"].sum()), "lượt fit CV; sau đó refit 8 nhóm thắng.")

,Tiền xử lý,Đặc trưng,Thuật toán,Cấu hình,Lượt fit CV
0,giữ stop words,BoW,Naive Bayes,6,30
1,giữ stop words,BoW,SVM,8,40
2,giữ stop words,TF-IDF,Naive Bayes,6,30
3,giữ stop words,TF-IDF,SVM,8,40
4,có loại stop words,BoW,Naive Bayes,6,30
5,có loại stop words,BoW,SVM,8,40
6,có loại stop words,TF-IDF,Naive Bayes,6,30
7,có loại stop words,TF-IDF,SVM,8,40


Tổng 56 cấu hình, 280 lượt fit CV; sau đó refit 8 nhóm thắng.


## 4. Chạy toàn bộ 56 cấu hình

Với **mỗi** tổ hợp n-gram và alpha/C, `GridSearchCV.fit` tạo năm bản Pipeline:
vectorizer học trên train của fold; NB/SVM fit vector và nhãn; sau đó dự đoán
validation. `cv_results_` giữ năm điểm và trung bình. `refit="f1"` fit lại
tổ hợp thắng trên toàn train. Tên `mean_test_f1` của thư viện chỉ **validation
CV** ở đây, không phải test độc lập.

Cell này chạy 280 lượt fit CV nên có thể mất vài phút.

In [4]:
rows, winners = [], []
with tempfile.TemporaryDirectory(prefix="cv_cache_", dir=OUT) as cache_dir:
    cache = Memory(cache_dir, verbose=0)
    for pre, series in TEXT.items():
        X = series.iloc[train_idx].to_numpy()
        for feature in FEATURES:
            for algorithm in MODELS:
                pipe, grid = audit_pipe(feature, algorithm, cache)
                search = GridSearchCV(pipe, grid, scoring=CV_SCORING,
                                      cv=FOLDS, n_jobs=N_JOBS,
                                      refit="f1", error_score="raise")
                search.fit(X, y_train)
                cv = search.cv_results_
                for j, params in enumerate(cv["params"]):
                    row = {"Tiền xử lý": pre, "Đặc trưng": feature,
                           "Thuật toán": algorithm,
                           "N-gram": str(params["vec__ngram_range"]),
                           "Tham số": (f"alpha={params['clf__alpha']}"
                                      if algorithm == "Naive Bayes"
                                      else f"C={params['clf__C']}"),
                           "class_weight": ("" if algorithm == "Naive Bayes"
                                            else str(params["clf__class_weight"])),
                           "CV Precision": float(cv["mean_test_precision"][j]),
                           "CV Recall": float(cv["mean_test_recall"][j]),
                           "CV F1": float(cv["mean_test_f1"][j]),
                           "F1 std": float(cv["std_test_f1"][j]),
                           "Hạng trong nhóm": int(cv["rank_test_f1"][j])}
                    for fold in range(len(FOLDS)):
                        row[f"F1 fold {fold+1}"] = float(cv[f"split{fold}_test_f1"][j])
                    rows.append(row)
                j = search.best_index_
                winners.append({"Tiền xử lý": pre, "Đặc trưng": feature,
                                "Thuật toán": algorithm,
                                "CV F1 tốt nhất": float(cv["mean_test_f1"][j]),
                                "N-gram thắng": str(search.best_params_["vec__ngram_range"]),
                                "Tham số thắng": (f"alpha={search.best_params_['clf__alpha']}"
                                                  if algorithm == "Naive Bayes"
                                                  else f"C={search.best_params_['clf__C']}")})
                print("Xong:", pre, feature, algorithm,
                      "CV F1 =", round(float(cv["mean_test_f1"][j]), 4), flush=True)

ALL = pd.DataFrame(rows)
WINNERS = pd.DataFrame(winners)
assert len(ALL) == int(plan["Cấu hình"].sum()) and ALL["CV F1"].notna().all()
ALL.to_csv(OUT / "all_cv_configs.csv", index=False, encoding="utf-8-sig")
WINNERS.to_csv(OUT / "best_by_group.csv", index=False, encoding="utf-8-sig")
print("Đã lưu:", OUT / "all_cv_configs.csv", OUT / "best_by_group.csv")

Xong: giữ stop words BoW Naive Bayes CV F1 = 0.6077


Xong: giữ stop words BoW SVM CV F1 = 0.7087


Xong: giữ stop words TF-IDF Naive Bayes CV F1 = 0.5959


Xong: giữ stop words TF-IDF SVM CV F1 = 0.7736


Xong: có loại stop words BoW Naive Bayes CV F1 = 0.5859


Xong: có loại stop words BoW SVM CV F1 = 0.6305


Xong: có loại stop words TF-IDF Naive Bayes CV F1 = 0.5835


Xong: có loại stop words TF-IDF SVM CV F1 = 0.7562


Đã lưu: C:\KH086\PT&KP DLVB\PROJECT\ptit-facebook-spam-ham\artifacts\grid_audit\all_cv_configs.csv C:\KH086\PT&KP DLVB\PROJECT\ptit-facebook-spam-ham\artifacts\grid_audit\best_by_group.csv


## 5. Xem từng điểm và dựng lại B1–B4

Bảng đầu có một hàng cho mỗi bộ tham số; `F1 fold 1–5` là điểm validation
từng lượt, `CV F1` là trung bình. Bảng thứ hai có 8 nhóm thắng (hai nhánh
từ dừng × bốn cặp đặc trưng/thuật toán). Chọn nhánh chứa CV F1 cao nhất,
rồi gắn B1–B4 theo thứ tự BoW+NB, BoW+SVM, TF-IDF+NB, TF-IDF+SVM.

Đây là lựa chọn **trong train**. Notebook 02 mới tiếp tục khảo sát S1–S3
và cuối cùng đánh giá trên test độc lập.

In [5]:
columns = ["Tiền xử lý", "Đặc trưng", "Thuật toán", "N-gram",
           "Tham số", "class_weight", "CV Precision", "CV Recall",
           "CV F1", "F1 std", "Hạng trong nhóm",
           *[f"F1 fold {i+1}" for i in range(len(FOLDS))]]
display(ALL[columns].round(4))
display(WINNERS.round(4))
main_pre = WINNERS.loc[WINNERS["CV F1 tốt nhất"].idxmax(), "Tiền xử lý"]
BASELINES = WINNERS[WINNERS["Tiền xử lý"] == main_pre].copy()
order = [(feature, algorithm) for feature in FEATURES for algorithm in MODELS]
BASELINES["Mã"] = BASELINES.apply(
    lambda r: "B" + str(order.index((r["Đặc trưng"], r["Thuật toán"])) + 1),
    axis=1)
BASELINES = BASELINES.sort_values("Mã")
display(BASELINES[["Mã", "Đặc trưng", "Thuật toán",
                   "CV F1 tốt nhất", "N-gram thắng", "Tham số thắng"]].round(4))
leader = BASELINES.loc[BASELINES["CV F1 tốt nhất"].idxmax()]
print("Nhánh được chọn:", main_pre, "| cơ sở dẫn đầu:", leader["Mã"],
      "| CV F1:", round(leader["CV F1 tốt nhất"], 4))

,Tiền xử lý,Đặc trưng,Thuật toán,N-gram,Tham số,class_weight,CV Precision,CV Recall,CV F1,F1 std,Hạng trong nhóm,F1 fold 1,F1 fold 2,F1 fold 3,F1 fold 4,F1 fold 5
0,giữ stop words,BoW,Naive Bayes,"(1, 1)",alpha=0.1,,0.6881,0.4983,0.5730,0.0721,6,0.5818,0.4379,0.6445,0.6230,0.5777
1,giữ stop words,BoW,Naive Bayes,"(1, 2)",alpha=0.1,,0.7103,0.5284,0.6012,0.0812,3,0.6194,0.4410,0.6607,0.6479,0.6371
2,giữ stop words,BoW,Naive Bayes,"(1, 1)",alpha=0.3,,0.6900,0.5032,0.5768,0.0728,4,0.5800,0.4396,0.6447,0.6308,0.5889
3,giữ stop words,BoW,Naive Bayes,"(1, 2)",alpha=0.3,,0.7091,0.5297,0.6019,0.0822,2,0.6206,0.4397,0.6624,0.6479,0.6390
4,giữ stop words,BoW,Naive Bayes,"(1, 1)",alpha=1.0,,0.6968,0.4958,0.5737,0.0725,5,0.5845,0.4370,0.6424,0.6259,0.5786
5,giữ stop words,BoW,Naive Bayes,"(1, 2)",alpha=1.0,,0.7582,0.5153,0.6077,0.0825,1,0.6191,0.4462,0.6720,0.6530,0.6484
6,giữ stop words,BoW,SVM,"(1, 1)",C=0.1,None,0.8343,0.4540,0.5808,0.0820,6,0.5911,0.4270,0.6704,0.6211,0.5944
7,giữ stop words,BoW,SVM,"(1, 2)",C=0.1,None,0.8376,0.4653,0.5916,0.0806,5,0.6161,0.4360,0.6685,0.6309,0.6066
8,giữ stop words,BoW,SVM,"(1, 1)",C=0.3,None,0.7584,0.4614,0.5667,0.0760,7,0.5812,0.4245,0.6531,0.5956,0.5791
9,giữ stop words,BoW,SVM,"(1, 2)",C=0.3,None,0.7770,0.6011,0.6720,0.0745,3,0.6894,0.5249,0.7270,0.7057,0.7128


,Tiền xử lý,Đặc trưng,Thuật toán,CV F1 tốt nhất,N-gram thắng,Tham số thắng
0,giữ stop words,BoW,Naive Bayes,0.6077,"(1, 2)",alpha=1.0
1,giữ stop words,BoW,SVM,0.7087,"(1, 2)",C=1.0
2,giữ stop words,TF-IDF,Naive Bayes,0.5959,"(1, 2)",alpha=0.1
3,giữ stop words,TF-IDF,SVM,0.7736,"(1, 2)",C=0.3
4,có loại stop words,BoW,Naive Bayes,0.5859,"(1, 2)",alpha=1.0
5,có loại stop words,BoW,SVM,0.6305,"(1, 2)",C=1.0
6,có loại stop words,TF-IDF,Naive Bayes,0.5835,"(1, 2)",alpha=0.1
7,có loại stop words,TF-IDF,SVM,0.7562,"(1, 2)",C=1.0


,Mã,Đặc trưng,Thuật toán,CV F1 tốt nhất,N-gram thắng,Tham số thắng
0,B1,BoW,Naive Bayes,0.6077,"(1, 2)",alpha=1.0
1,B2,BoW,SVM,0.7087,"(1, 2)",C=1.0
2,B3,TF-IDF,Naive Bayes,0.5959,"(1, 2)",alpha=0.1
3,B4,TF-IDF,SVM,0.7736,"(1, 2)",C=0.3


Nhánh được chọn: giữ stop words | cơ sở dẫn đầu: B4 | CV F1: 0.7736


## 6. So n-gram trên cùng alpha/C

Bảng `PAIRS` giữ nguyên từ dừng, đặc trưng, thuật toán và alpha/C;
chỉ thay `(1,1)` ↔ `(1,2)`. Chênh dương là `(1,2)` cao hơn.
Bảng `BEST_NGRAM` cho điểm tốt nhất của mỗi n-gram sau khi chọn alpha/C
**riêng bên trong n-gram đó**. CV cho thấy cái nào tốt hơn trên dữ liệu này;
không chứng minh nó luôn hơn trên dữ liệu Facebook khác.

In [6]:
PAIRS = ALL.pivot(index=["Tiền xử lý", "Đặc trưng", "Thuật toán", "Tham số"],
                  columns="N-gram", values="CV F1").reset_index()
PAIRS["Chênh (1,2) - (1,1)"] = PAIRS["(1, 2)"] - PAIRS["(1, 1)"]
display(PAIRS.round(4))
best_idx = ALL.groupby(["Tiền xử lý", "Đặc trưng", "Thuật toán",
                        "N-gram"])["CV F1"].idxmax()
BEST_NGRAM = ALL.loc[best_idx, ["Tiền xử lý", "Đặc trưng", "Thuật toán",
                                "N-gram", "Tham số", "CV F1"]]
display(BEST_NGRAM.sort_values(["Tiền xử lý", "Đặc trưng",
                                "Thuật toán", "N-gram"]).round(4))
print("(1,2) thắng", int((PAIRS["Chênh (1,2) - (1,1)"] > 0).sum()),
      "trên", len(PAIRS), "cặp cùng alpha/C; được chọn ở",
      int(WINNERS["N-gram thắng"].eq("(1, 2)").sum()),
      "trên", len(WINNERS), "nhóm.")

N-gram,Tiền xử lý,Đặc trưng,Thuật toán,Tham số,"(1, 1)","(1, 2)","Chênh (1,2) - (1,1)"
0,có loại stop words,BoW,Naive Bayes,alpha=0.1,0.5681,0.5805,0.0123
1,có loại stop words,BoW,Naive Bayes,alpha=0.3,0.5711,0.5855,0.0144
2,có loại stop words,BoW,Naive Bayes,alpha=1.0,0.5728,0.5859,0.0131
3,có loại stop words,BoW,SVM,C=0.1,0.5692,0.5785,0.0093
4,có loại stop words,BoW,SVM,C=0.3,0.5591,0.5674,0.0084
5,có loại stop words,BoW,SVM,C=1.0,0.5369,0.6305,0.0936
6,có loại stop words,BoW,SVM,C=3.0,0.5200,0.6091,0.0892
7,có loại stop words,TF-IDF,Naive Bayes,alpha=0.1,0.5447,0.5835,0.0388
8,có loại stop words,TF-IDF,Naive Bayes,alpha=0.3,0.5430,0.5759,0.0329
9,có loại stop words,TF-IDF,Naive Bayes,alpha=1.0,0.5031,0.5100,0.0069


,Tiền xử lý,Đặc trưng,Thuật toán,N-gram,Tham số,CV F1
32,có loại stop words,BoW,Naive Bayes,"(1, 1)",alpha=1.0,0.5728
33,có loại stop words,BoW,Naive Bayes,"(1, 2)",alpha=1.0,0.5859
34,có loại stop words,BoW,SVM,"(1, 1)",C=0.1,0.5692
39,có loại stop words,BoW,SVM,"(1, 2)",C=1.0,0.6305
42,có loại stop words,TF-IDF,Naive Bayes,"(1, 1)",alpha=0.1,0.5447
43,có loại stop words,TF-IDF,Naive Bayes,"(1, 2)",alpha=0.1,0.5835
50,có loại stop words,TF-IDF,SVM,"(1, 1)",C=0.3,0.6657
53,có loại stop words,TF-IDF,SVM,"(1, 2)",C=1.0,0.7562
2,giữ stop words,BoW,Naive Bayes,"(1, 1)",alpha=0.3,0.5768
5,giữ stop words,BoW,Naive Bayes,"(1, 2)",alpha=1.0,0.6077


(1,2) thắng 27 trên 28 cặp cùng alpha/C; được chọn ở 8 trên 8 nhóm.


## 7. Cách tự chạy và đổi lưới

1. Trong VS Code chọn kernel `.venv/Scripts/python.exe` rồi **Run All**.
2. `plan` cho số cấu hình; `ALL` cho **từng điểm CV**; `WINNERS`
   cho tám ứng viên thắng; `BASELINES` cho B1–B4.
3. Muốn thử thêm giá trị, sửa `AUDIT_NGRAMS`, `AUDIT_ALPHAS` hoặc
   `AUDIT_CS` ở mục 3 và chạy lại từ đó đến cuối. Thí dụ
   `AUDIT_CS = [0.1, 0.3, 0.5, 1.0, 3.0]` thêm mức C=0.5.
4. Mở `artifacts/grid_audit/all_cv_configs.csv` để lọc mọi hàng và
   `best_by_group.csv` để xem tám nhóm. Không dùng test để điều chỉnh lưới.

Điểm có thể lệch rất nhỏ ở chữ số cuối giữa các lần chạy; dùng cùng dữ liệu,
phiên bản thư viện và phép chia để so sánh. Tham khảo
[GridSearchCV](https://scikit-learn.org/1.6/modules/generated/sklearn.model_selection.GridSearchCV.html)
và [ParameterGrid](https://scikit-learn.org/1.6/modules/generated/sklearn.model_selection.ParameterGrid.html).